In [ ]:
import json
from pathlib import Path

from conversion_helpers import (
    add_relative_ou_stimulus,
    add_spike_replay,
    add_soma_reports,
    add_connection_overrides,
    add_conditions,
)
CURATED_PATH = Path("/Users/james/Documents/obi/Data/Simulations/BBP-curated/4_spikesorting_stimulus_test_neuropixels_8-1-24__8slc_80f_360r_50t_200ms_1_smallest_fiber_gids")
RAW_CONFIGS_PATH = Path("/Users/james/Documents/obi/Data/Simulations/BBP-raw/4_spikesorting_stimulus_test_neuropixels_8-1-24__8slc_80f_360r_50t_200ms_1_smallest_fiber_gids/")
RAW_INPUT_OUTPUT_PATH = RAW_CONFIGS_PATH / "0fcb7709-b1e9-4d84-b056-5801f20d55af"

sim_length = 101500
ou_start_time = 250
ca_conc = 1.05

for index in range(36):

    config = {
        "version": 1,
        "run": {
            "dt": 0.025,
            "random_seed": 1,
            "tstop": sim_length
        },
        "spike_location": "AIS",
        "network": "../circuit_config.json",
        "node_set": "cyl_hex0_0.045",
        "node_set_file": "node_sets.json",
        "target_simulator": "CORENEURON",
        "output": {
            "output_dir": "./reporting",
            "spikes_file": "spikes.h5"
        },
        "inputs": {},
        "reports": {}
    }

    add_relative_ou_stimulus(config, "Stimulus gExc_L1", "Layer1", 2.993, 1.197, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L23E", "Layer23Excitatory", 18.183, 7.273, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L23I", "Layer23Inhibitory", 2.587, 1.035, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L4E", "Layer4Excitatory", 8.844, 3.538, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L4I", "Layer4Inhibitory", 3.13, 1.252, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L5E", "Layer5Excitatory", 16.394, 6.558, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L5I", "Layer5Inhibitory", 4.761, 1.904, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L6E", "Layer6Excitatory", 2.562, 1.025, ou_start_time, sim_length)
    add_relative_ou_stimulus(config, "Stimulus gExc_L6I", "Layer6Inhibitory", 3.026, 1.21, ou_start_time, sim_length)

    add_spike_replay(config, "VPM", sim_length, "input_VPM.h5", "proj_Thalamocortical_VPM_Source")
    # add_spike_replay(config, "POm", sim_length, "input_POM.h5", "proj_Thalamocortical_POM_Source")
    # add_soma_reports(config, sim_length)
    add_connection_overrides(config)
    add_conditions(config, ca_conc)

    curated_sim_dir = CURATED_PATH / str(index)
    curated_sim_dir.mkdir(exist_ok=True, parents=True)
    config_path = curated_sim_dir / Path(f"simulation_config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)
    

In [ ]:
RAW_TARGET_FILE = RAW_CONFIGS_PATH / "user.target"
ID_MAPPING_FILE = Path("/Users/james/Documents/obi/Data/Circuits/nbS1-O1/id_mapping.json")
NODE_SET_FILE = CURATED_PATH / "cyl_hex0_0.045_node_set.json"
TARGET_NAME = "cyl_hex0_0.045"
POPULATION = "S1nonbarrel_neurons"

# Extract GIDs from the named BBP target block (tokens of form `aN`)
lines = RAW_TARGET_FILE.read_text().splitlines()
start = next(i for i, ln in enumerate(lines) if ln.strip().startswith(f"Target Cell {TARGET_NAME}"))
open_i = next(j for j in range(start, len(lines)) if "{" in lines[j])
close_i = next(j for j in range(open_i + 1, len(lines)) if "}" in lines[j])
gids = [
    int(tok[1:])
    for ln in lines[open_i + 1:close_i]
    for tok in ln.split()
    if tok.startswith("a") and tok[1:].isdigit()
]
print(f"{TARGET_NAME}: {len(gids)} GIDs ({len(set(gids))} unique)")

# BBP `aN` is 1-indexed GID, SONATA node_id is 0-indexed: old_id = N - 1
pop_map = json.loads(ID_MAPPING_FILE.read_text())[POPULATION]
old_to_new = dict(zip(pop_map["old_id"], pop_map["new_id"]))

new_ids, unmapped = [], []
for gid in gids:
    old_id = gid - 1
    if old_id in old_to_new:
        new_ids.append(int(old_to_new[old_id]))
    else:
        unmapped.append(gid)
new_ids = sorted(set(new_ids))
print(f"mapped: {len(new_ids)} unique new_ids; unmapped: {len(unmapped)} GIDs")

node_set = {TARGET_NAME: {"population": POPULATION, "node_id": new_ids}}
NODE_SET_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(NODE_SET_FILE, "w") as f:
    json.dump(node_set, f)
print(f"wrote {NODE_SET_FILE}")

In [ ]:
import h5py
import numpy as np

SPIKES_FILE = CURATED_PATH / "1" / "reporting" / "spikes.h5"

with h5py.File(SPIKES_FILE, "r") as f:
    spiking_node_ids = np.unique(f[f"spikes/{POPULATION}/node_ids"][:])
print(f"unique spiking node_ids: {len(spiking_node_ids)}")

node_set_ids = set(new_ids)
missing = [int(nid) for nid in spiking_node_ids if int(nid) not in node_set_ids]
print(f"spiking node_ids not in node set: {len(missing)}")
if missing:
    print(f"example missing: {missing[:10]}")
else:
    print(f"all {len(spiking_node_ids)} spiking node_ids are in {TARGET_NAME}")